In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

import numpy as np
import pandas as pd
from typing import List, Tuple



C:\Users\raavi\AppData\Local\Temp\ipykernel_27168\4177575985.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:

## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [6]:
### save sample documents to text files

import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(os.path.join(temp_dir, f"doc_{i+1}.txt"), "w") as f:
        f.write(doc)
print(f"Sample documents saved to: {temp_dir}")


Sample documents saved to: C:\Users\raavi\AppData\Local\Temp\tmpvz4v7ka_


In [7]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"doc_{i+1}.txt", "w") as f:
        f.write(doc)
print(f"Sample documents saved to: {temp_dir}")


Sample documents saved to: C:\Users\raavi\AppData\Local\Temp\tmpaxj0t7hw


### 2. Document loading

In [8]:
pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "Data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"/n first document content: {documents[0].page_content[:200]}...")

Loaded 3 documents
/n first document content: 
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. Ther...


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks= text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)},from {len(documents)} original documents")


Total chunks created: 7,from 3 original documents


In [11]:
chunks

[Document(metadata={'source': 'Data\\doc_1.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'Data\\doc_1.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'Data\\doc_1.txt'}, page_content='interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'Data\\doc_2.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'Data\\doc_2.txt'}, page_content='Deep learning is a subset of machine learning based on artificial neural networks. \n    Thes

In [12]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [13]:
#### Embeddings and Vector Store Creation

sample_text = "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed."
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001BEC7EEA930>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001BEC7EEBC20>, model='text-embedding-3-small', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [14]:
vector = embeddings.embed_query(sample_text)
vector


[-0.0113983154296875,
 -0.0211639404296875,
 -0.0007700920104980469,
 -0.027008056640625,
 0.05853271484375,
 0.007049560546875,
 0.036376953125,
 0.035552978515625,
 0.0198822021484375,
 0.08013916015625,
 0.022308349609375,
 -0.0132598876953125,
 -0.041717529296875,
 -0.0202484130859375,
 0.006435394287109375,
 0.040252685546875,
 0.0021839141845703125,
 0.01251220703125,
 0.052215576171875,
 -0.01206207275390625,
 0.024078369140625,
 0.016357421875,
 0.03741455078125,
 0.0189666748046875,
 0.04443359375,
 -0.061004638671875,
 -0.004673004150390625,
 0.0236053466796875,
 -0.0301666259765625,
 0.00580596923828125,
 1.1146068572998047e-05,
 -0.005615234375,
 -0.045196533203125,
 -0.007282257080078125,
 0.04742431640625,
 0.043243408203125,
 0.01194000244140625,
 -0.04150390625,
 -0.0249786376953125,
 0.038177490234375,
 -0.0292205810546875,
 -0.0242767333984375,
 0.012542724609375,
 0.07269287109375,
 -0.03387451171875,
 -0.0135345458984375,
 -0.0233612060546875,
 -0.01910400390625,
 0

In [15]:
print(f"Vector length: {len(vector)}")

Vector length: 1024


### Intialize the chromaDB

In [16]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from langchain_community.vectorstores import Chroma

persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="deep_learning_test_v2"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print("Vector store created and persisted to disk.")
print(f"Vector store directory: {persist_directory}")

Vector store created with 21 vectors
Vector store created and persisted to disk.
Vector store directory: ./chroma_db


In [18]:
### Test similarity search

query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'Data\\doc_2.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'Data\\doc_2.txt'}, page_content='Deep Learning and Neural Networks'),
 Document(metadata={'source': 'Data\\doc_2.txt'}, page_content='Deep Learning and Neural Networks')]

In [19]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is Deep Learning?

Top 3 similar chunks:

--- Chunk 1 ---
Deep Learning and Neural Networks...
Source: Data\doc_2.txt

--- Chunk 2 ---
Deep Learning and Neural Networks...
Source: Data\doc_2.txt

--- Chunk 3 ---
Deep Learning and Neural Networks...
Source: Data\doc_2.txt


In [20]:
vectorstore.similarity_search_with_score(query,k=3)
print(f"Similarity search with scores for query: {query}")
results = vectorstore.similarity_search_with_score(query, k=3)
for i, (doc, score) in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Similarity Score: {score:.4f}")

Similarity search with scores for query: what is Deep Learning?

--- Chunk 1 ---
Deep Learning and Neural Networks...
Source: Data\doc_2.txt
Similarity Score: 0.6294

--- Chunk 2 ---
Deep Learning and Neural Networks...
Source: Data\doc_2.txt
Similarity Score: 0.6294

--- Chunk 3 ---
Deep Learning and Neural Networks...
Source: Data\doc_2.txt
Similarity Score: 0.6295


### Intialize LLM, RAG Chain, Prompt Template

In [21]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.2, max_tokens=500)

In [22]:
llm.invoke("What is Gen ai?")

AIMessage(content='Gen AI refers to the next generation of artificial intelligence that is designed to be more advanced, adaptive, and human-like in its capabilities. It aims to create AI systems that can learn, reason, and interact with humans in a more natural and intuitive way. Gen AI is expected to have a wide range of applications in various industries, including healthcare, finance, education, and entertainment.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 12, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DmNL1oHeqiqZahb4qekgPeeOOg1qU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e8969-

In [23]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(model="gpt-3.5-turbo", temperature=0.2, max_tokens=500)
llm.invoke("What is Gen ai?")
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001BECC70F9B0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001BECC7550A0>, root_client=<openai.OpenAI object at 0x000001BEC62B1490>, root_async_client=<openai.AsyncOpenAI object at 0x000001BECC70D820>, temperature=0.2, model_kwargs={}, o

In [24]:
pip install langchain-community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import sys
!{sys.executable} -m pip install -U langchain langchain-core langchain-community


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import sys
!{sys.executable} -m pip show langchain
!{sys.executable} -m pip show langchain-core
!{sys.executable} -m pip show langchain-community

Name: langchain
Version: 1.3.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\Users\raavi\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-core
Version: 1.4.0
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\Users\raavi\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-huggingface, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langc

In [29]:
pip uninstall -y langchain langchain-core langchain-community

Found existing installation: langchain 1.3.2
Uninstalling langchain-1.3.2:
  Successfully uninstalled langchain-1.3.2
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Note: you may need to restart the kernel to use updated packages.


In [30]:
!pip install langchain==0.2.16
!pip install langchain-core==0.2.38
!pip install langchain-community==0.2.16

  Using cached langchain-0.2.16-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
Using cached langchain-0.2.16-py3-none-any.whl (1.0 MB)
Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
Using cached langchain_text_splitters-0.2.4-py3-none-any.whl (25 kB)
Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)

  Attempting uninstall: langsmith

    Found existing installation: langsmith 0.8.8

    Uninstalling langsmith-0.8.8:

      Successfully uninstalled langsmith-0.8.8

   ---------------------------------------- 0/4 [langsmith]
   ---------------------------------------- 0/4 [langsmith]
   ---------------------------------------- 0/4 [langsmith]
   ---------------------------------------- 0/4 [langsmith]
   ---------------------------------------- 0/4 [langsmith]
 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.2.43 which is incompatible.
langchain-openai 1.2.2 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langgraph 1.2.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.43 which is incompatible.


  Using cached langchain_core-0.2.38-py3-none-any.whl.metadata (6.2 kB)
Using cached langchain_core-0.2.38-py3-none-any.whl (396 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.2.43
    Uninstalling langchain-core-0.2.43:
      Successfully uninstalled langchain-core-0.2.43


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.2.38 which is incompatible.
langchain-classic 1.0.7 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.2.38 which is incompatible.
langchain-openai 1.2.2 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.2.38 which is incompatible.
langgraph 1.2.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.2.38 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.2.38 which is incompatible.


  Using cached langchain_community-0.2.16-py3-none-any.whl.metadata (2.7 kB)
Using cached langchain_community-0.2.16-py3-none-any.whl (2.3 MB)


In [ ]:
pip uninstall -y langchain langchain-core langchain-community langchain-openai pydantic


Found existing installation: langchain 0.2.16
Uninstalling langchain-0.2.16:
  Successfully uninstalled langchain-0.2.16
Found existing installation: langchain-core 0.2.38
Uninstalling langchain-core-0.2.38:
  Successfully uninstalled langchain-core-0.2.38
Found existing installation: langchain-community 0.2.16
Uninstalling langchain-community-0.2.16:
  Successfully uninstalled langchain-community-0.2.16
Found existing installation: langchain-openai 0.1.23
Uninstalling langchain-openai-0.1.23:
  Successfully uninstalled langchain-openai-0.1.23
Found existing installation: pydantic 2.8.2
Uninstalling pydantic-2.8.2:
  Successfully uninstalled pydantic-2.8.2
Note: you may need to restart the kernel to use updated packages.


In [39]:
pip install -U langchain langchain-community langchain-openai pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import langchain
import langchain_core
import pydantic

print(langchain.__version__)
print(langchain_core.__version__)
print(pydantic.__version__)

1.3.2
1.4.0
2.13.4


In [42]:
!pip install -U langchain-classic

In [44]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

print("Imports working")

Imports working


In [46]:
retriver = vectorstore.as_retriever(
    search_kwargs={"k": 3},
)
retriver

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001BEC87E25A0>, search_kwargs={'k': 3})

In [ ]:
### create a prompt template for the retrieval chain

system_prompt = """You are an assistant that helps answer questions about machine learning concepts based on retrieved document chunks.
Use the following retrieved chunks to answer the question. If the retrieved information is insufficient, say you don't know.
{retrieved_chunks}

context: {context} """

prompt = ChatPromptTemplate.from_messages(
    (("system", system_prompt), 
     ("human", "{question}"))
)


In [48]:
prompt

ChatPromptTemplate(input_variables=['context', 'question', 'retrieved_chunks'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'retrieved_chunks'], input_types={}, partial_variables={}, template="You are an assistant that helps answer questions about machine learning concepts based on retrieved document chunks.\nUse the following retrieved chunks to answer the question. If the retrieved information is insufficient, say you don't know.\n{retrieved_chunks}\n\ncontext: {context} "), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [49]:
### create retrieval chain

documents_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
documents_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'question', 'retrieved_chunks'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'retrieved_chunks'], input_types={}, partial_variables={}, template="You are an assistant that helps answer questions about machine learning concepts based on retrieved document chunks.\nUse the following retrieved chunks to answer the question. If the retrieved information is insufficient, say you don't know.\n{retrieved_chunks}\n\ncontext: {context} "), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo

In [53]:
rag_chain = create_retrieval_chain(
    retriever=retriver,
    combine_docs_chain=documents_chain
)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001BEC87E25A0>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'question', 'retrieved_chunks'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'retrieved_chunks'], input_types={}, partial_variables={}, template="You are an assistant that helps answer questions about machine learning concepts based 

In [61]:
prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context:
{context}

Question:
{input}
""")

In [62]:
documents_chain = create_stuff_documents_chain(llm, prompt)

rag_chain = create_retrieval_chain(
    retriever=retriver,  # or retriever
    combine_docs_chain=documents_chain
)

In [63]:
response = rag_chain.invoke({
    "input": "What is Deep Learning?"
})

print(response["answer"])

Deep learning is a subset of machine learning that uses artificial neural networks to model and solve complex problems. It involves training these networks on large amounts of data to learn patterns and make predictions. Deep learning has been successful in various applications such as image and speech recognition.


In [64]:
# Function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Using create_retrieval_chain approach
    result = rag_chain.invoke({"input": question})
    
    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
    
    return result

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

Question: What are the three types of machine learning?
--------------------------------------------------
Answer: The three types of machine learning are supervised learning, unsupervised learning, and reinforcement learning.

Retrieved Context:

--- Source 1 ---
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine l...

--- Source 2 ---
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine l...

--- Source 3 ---
Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine l...


Question: What is deep learning and how does it relate to neural networks?
----------

### Create RAG chain Alternative- Using LCEL 

In [66]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [67]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [68]:
retriver

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001BEC87E25A0>, search_kwargs={'k': 3})

In [70]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [71]:
## Build the chain ussing LCEL
rag_chain_lcel=(
    { 
        "context":retriver | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001BEC87E25A0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_in

In [74]:
response = rag_chain_lcel.invoke("what is Deep Learning?")
print(response)

Deep Learning is a type of machine learning that uses neural networks with multiple layers to analyze and learn from data. It is a subset of artificial intelligence that focuses on learning representations of data through multiple layers of processing.


In [76]:
retriver.get_relevant_documents("What is Deep Learning?")

AttributeError: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'

In [79]:
def query_rag_lcel(question):
    result = rag_chain_lcel.invoke(question)

    print(f"Answer: {result}")

    docs = retriver.invoke(question)

    print("\nRetrieved Context:")
    for i, doc in enumerate(docs):
        print(f"\n--- Document {i+1} ---")
        print(doc.page_content[:500])
        print(doc.metadata)
    return result

In [80]:
print("Prompt template created")
query_rag_lcel("What is deep learning and how does it relate to neural networks?")

Prompt template created
Answer: Deep learning is a subset of machine learning that uses artificial neural networks to model and solve complex problems. Neural networks are a key component of deep learning, as they are designed to mimic the structure and function of the human brain to process and learn from data. In deep learning, multiple layers of neural networks are used to extract higher-level features from raw data, allowing for more accurate and sophisticated learning and decision-making processes.

Retrieved Context:

--- Document 1 ---
Deep Learning and Neural Networks
{'source': 'Data\\doc_2.txt'}

--- Document 2 ---
Deep Learning and Neural Networks
{'source': 'Data\\doc_2.txt'}

--- Document 3 ---
Deep Learning and Neural Networks
{'source': 'Data\\doc_2.txt'}


'Deep learning is a subset of machine learning that uses artificial neural networks to model and solve complex problems. Neural networks are a key component of deep learning, as they are designed to mimic the structure and function of the human brain to process and learn from data. In deep learning, multiple layers of neural networks are used to extract higher-level features from raw data, allowing for more accurate and sophisticated learning and decision-making processes.'

###  Adding New Documents to Existing Vector Store

In [81]:
vectorstore


In [82]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [85]:
new_doc = Document(
    page_content=new_document,
    metadata={"source": "new_doc.txt",
              "category": "Reinforcement Learning"
              }
)
new_doc

Document(metadata={'source': 'new_doc.txt', 'category': 'Reinforcement Learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [89]:
new_chunks = text_splitter.split_documents([new_doc])
new_chunks


[Document(metadata={'source': 'new_doc.txt', 'category': 'Reinforcement Learning'}, page_content='Reinforcement Learning in Detail'),
 Document(metadata={'source': 'new_doc.txt', 'category': 'Reinforcement Learning'}, page_content='Reinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and'),
 Document(metadata={'source': 'new_doc.txt', 'category': 'Reinforcement Learning'}, page_content='Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [90]:
### Add the new document to the vector store
vectorstore.add_documents(new_chunks)


['ba9e5df7-c3e5-41e5-a6f3-1c245d72fec7',
 'a2fed4cd-be06-4478-b0b6-9f7a50c5a0cb',
 '2e86cdda-9085-47f0-88d9-1e01362a035d']

In [93]:
print(f"New document added. Total vectors in store: {vectorstore._collection.count()}")
print(f"new chunks added: {len(new_chunks)}")
print(f"old chunks: {len(chunks)}")

New document added. Total vectors in store: 24
new chunks added: 3
old chunks: 7


In [94]:
new_question = "What is reinforcement learning and how does it differ from supervised learning?"
query_rag_lcel(new_question)

Answer: Reinforcement learning is a type of machine learning where an agent learns to make decisions by interacting with an environment and receives rewards or penalties based on its actions to maximize cumulative reward over time. It differs from supervised learning in that supervised learning uses labeled data to train models, while reinforcement learning learns through interacting with an environment and receiving rewards or penalties based on its actions.

Retrieved Context:

--- Document 1 ---
Reinforcement Learning in Detail
{'category': 'Reinforcement Learning', 'source': 'new_doc.txt'}

--- Document 2 ---
Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, D

'Reinforcement learning is a type of machine learning where an agent learns to make decisions by interacting with an environment and receives rewards or penalties based on its actions to maximize cumulative reward over time. It differs from supervised learning in that supervised learning uses labeled data to train models, while reinforcement learning learns through interacting with an environment and receiving rewards or penalties based on its actions.'

In [97]:

from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [98]:
from langchain.chains import create_history_aware_retriever

ModuleNotFoundError: No module named 'langchain.chains.history_aware_retriever'